In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

sys.path.append(os.path.abspath(".."))

from src.data import get_xy
from src import models as mod
from src import visualization as viz

In [ ]:
df_train = pd.read_excel("training_barcelona_shots.xlsx", header=0)
df_test = pd.read_excel("testing_barcelona_shots.xlsx", header=0)

df_train = df_train[df_train["pass_id"].notna()].copy()
df_test = df_test[df_test["pass_id"].notna()].copy()

pass_features = [
    "pass_start_distance",
    "pass_length",
    "pass_end_distance",
    "pass_angle",
    "cross",
    "cut_back",
    "pass_height_b",
    "pass_type_b"
]

X_train, y_train = get_xy(df_train, pass_features)
X_test, y_test = get_xy(df_test, pass_features)

# XGBoost Model

In [ ]:
model_xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 1, 5]
}

## Cross-validation and Hyperparameter Tuning

In [ ]:
grid_xgb, results_xgb = mod.evaluate_model(model_xgb, param_grid_xgb, X_train, y_train)
mod.print_results(results_xgb)

## Final Model Training

In [ ]:
final_model_xgb = XGBClassifier(**grid_xgb.best_params_, eval_metric="logloss", random_state=42)
final_model_xgb.fit(X_train, y_train)

y_pred_proba_xgb = final_model_xgb.predict_proba(X_test)[:, 1]

## Test Set Evaluation

In [ ]:
df_test["predicted_xg_xgb"] = y_pred_proba_xgb
mod.summarize_predictions(y_test, y_pred_proba_xgb)
mod.print_test_metrics(y_test, y_pred_proba_xgb)

## Feature Importance

In [ ]:
importances_df_xgb = mod.get_permutation_importance(final_model_xgb, X_test.columns, X_test, y_test)
viz.plot_feature_importances(importances_df_xgb, "XGBoost Classifier")

# Logistic Regression

## Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## L1 Regularization Path

In [ ]:
C_values = np.logspace(-6, 3, 50)

coefs = []

for C in C_values:
    lr = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=1000,
        C=C,
        random_state=42
    )
    lr.fit(X_train_scaled, y_train)
    coefs.append(lr.coef_[0])

coefs = np.array(coefs)
viz.plot_l1_paths(C_values, coefs, X_train.columns)

## L1 Hyperparameter Tuning

In [ ]:
model_lr = LogisticRegression(
    max_iter=1000, 
    random_state=42
)

param_grid_lr = {
    "penalty": ["l1"],
    "C": [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1],
    "solver": ["liblinear"]
}

In [ ]:
grid_lr, results_lr = mod.evaluate_model(model_lr, param_grid_lr, X_train_scaled, y_train)
mod.print_results(results_lr)

## L1 Coefficient Inspection

In [ ]:
final_log_reg = LogisticRegression(**grid_lr.best_params_, max_iter=1000, random_state=42)
final_log_reg.fit(X_train_scaled, y_train)

y_pred_proba_lr = final_log_reg.predict_proba(X_test_scaled)[:, 1]

In [ ]:
best_lr = grid_lr.best_estimator_
coef = best_lr.coef_[0]
mod.print_coefficients(X_train.columns, coef)

## Feature Selection from L1 Regularization

In [ ]:
pass_features_after_l1_regularization = [
    "pass_end_distance",
    "cut_back",
    "pass_height_b"
]

## L2 Model Construction

In [ ]:
X_train_l2, y_train_l2 = get_xy(df_train, pass_features_after_l1_regularization)
X_test_l2, y_test_l2 = get_xy(df_test, pass_features_after_l1_regularization)

X_train_scaled_lr_l2 = scaler.fit_transform(X_train_l2)
X_test_scaled_lr_l2 = scaler.transform(X_test_l2)

## L2 Hyperparameter Tuning

In [ ]:
model_lr_l2 = LogisticRegression(
    max_iter=1000, 
    random_state=42
)

param_grid_lr_l2 = {
    "penalty": ["l2"],
    "C": [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["liblinear"]
}

In [ ]:
grid_lr_l2, results_lr_l2 = mod.evaluate_model(model_lr_l2, param_grid_lr_l2, X_train_scaled_lr_l2, y_train_l2)
mod.print_results(results_lr_l2)

## L2 Coefficient Inspection

In [ ]:
best_lr_l2 = grid_lr_l2.best_estimator_
coef_l2 = best_lr_l2.coef_[0]
mod.print_coefficients(X_train_l2.columns, coef_l2)

## Final L2 Model Training

In [ ]:
final_log_reg_l2 = LogisticRegression(**grid_lr_l2.best_params_, max_iter=1000, random_state=42)
final_log_reg_l2.fit(X_train_scaled_lr_l2, y_train_l2)

y_pred_proba_lr_l2 = final_log_reg_l2.predict_proba(X_test_scaled_lr_l2)[:, 1]

## Test Set Evaluation

In [ ]:
df_test["predicted_xg_lr"] = y_pred_proba_lr_l2
mod.summarize_predictions(y_test, y_pred_proba_lr_l2)
mod.print_test_metrics(y_test, y_pred_proba_lr_l2)

## Bootstrap Coefficient Inference

In [ ]:
results = mod.bootstrap_logistic_inference(final_log_reg_l2, X_train_scaled_lr_l2, y_train, X_train_l2.columns)

print("\nL2-penalized Logistic Regression — Bootstrap Inference (2000 resamples)\n")
print(results.to_string(index=False))

## Feature Importance

In [ ]:
importances_df_lr = mod.get_permutation_importance(final_log_reg_l2, X_test_l2.columns, X_test_scaled_lr_l2, y_test)
viz.plot_feature_importances(importances_df_lr, "Logistic regression")

# Random Forest

In [ ]:
model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
}

## Cross-validation and Hyperparameter Tuning

In [ ]:
grid_rf, results_rf = mod.evaluate_model(model_rf, param_grid_rf, X_train, y_train)
mod.print_results(results_rf)

## Final Model Training

In [ ]:
final_model_rf = RandomForestClassifier(**grid_rf.best_params_, random_state=42, n_jobs=-1)
final_model_rf.fit(X_train, y_train)

y_pred_proba_rf = final_model_rf.predict_proba(X_test)[:, 1]

## Test Set Evaluation

In [ ]:
df_test["predicted_xg_rf"] = y_pred_proba_rf
mod.summarize_predictions(y_test, y_pred_proba_rf)
mod.print_test_metrics(y_test, y_pred_proba_rf)

In [ ]:
importances_df_rf = mod.get_permutation_importance(final_model_rf, X_test.columns, X_test, y_test)
viz.plot_feature_importances(importances_df_rf, "Random Forest Classifier")

# Support Vector Machine

In [ ]:
model_svm = SVC(probability=True, random_state=42)

param_grid_svm = {
    "C": [0.01, 0.025, 0.05, 0.1],
    "gamma": ["scale", "auto", 0.01, 0.1, 1],
    "kernel": ["rbf", "poly"]
}

## Cross-validation and Hyperparameter Tuning

In [ ]:
grid_svm, results_svm = mod.evaluate_model(model_svm, param_grid_svm, X_train_scaled, y_train)
mod.print_results(results_svm)

## Final Model Training

In [ ]:
final_model_svm = SVC(**grid_svm.best_params_, probability=True, random_state=42)
final_model_svm.fit(X_train_scaled, y_train)

y_pred_proba_svm = final_model_svm.predict_proba(X_test_scaled)[:, 1]

## Test Set Evaluation

In [ ]:
df_test["predicted_xg_svm"] = y_pred_proba_svm
mod.summarize_predictions(y_test, y_pred_proba_svm)
mod.print_test_metrics(y_test, y_pred_proba_svm)

## Feature Importance

In [ ]:
importances_df_svm = mod.get_permutation_importance(final_model_svm, X_test.columns, X_test_scaled, y_test)
viz.plot_feature_importances(importances_df_svm, "Support Vector Machine")

# Model Comparison

## ROC Curve Comparison

In [ ]:
model_predictions = {
    "XGBoost": y_pred_proba_xgb,
    "Logistic Regression": y_pred_proba_lr,
    "Random Forest": y_pred_proba_rf,
    "SVM": y_pred_proba_svm,
}

viz.plot_model_roc_curves(y_test, model_predictions)

In [ ]:
viz.plot_precision_recall_curves(y_test, model_predictions)